# Offline Bottom Detection Methods

Comparison of the new offline (bidirectional) bottom detection techniques against the existing CRIMAC-ported ensemble. All methods are tested on 5 randomly selected echograms from the raw file collection.

In [ ]:
import sys
sys.path.insert(0, '../..')
from project_paths import REPO_ROOT

import numpy as np
import matplotlib.pyplot as plt

from raw_sensor_integration.data_loader import load_raw
from raw_sensor_integration import ping_parser
from raw_sensor_integration.bottom_detection import (
    detect_bottom_ensemble_pipeline,
)
from raw_sensor_integration.bottom_detection_offline import (
    detect_bottom_tvg,
    detect_bottom_dp,
    compute_phase_variance,
    refine_bottom_snake,
)

## Select 5 random files

In [ ]:
RAW_DIR = REPO_ROOT / 'Datasets/raw_files/drive-download-20260317T113707Z-1-001'
all_raw = sorted(RAW_DIR.glob('*.raw'))
print(f'Found {len(all_raw)} raw files')

rng = np.random.default_rng(seed=42)
n_files = min(5, len(all_raw))
selected_idx = rng.choice(len(all_raw), size=n_files, replace=False)
selected_files = [all_raw[i] for i in sorted(selected_idx)]

for f in selected_files:
    print(f'  {f.name}')

## Load echograms and complex samples

In [ ]:
datasets = []

for raw_path in selected_files:
    print(f'Loading {raw_path.name}...')
    data = load_raw(raw_path, frequency=38000)

    # Also extract complex samples for phase analysis
    pings = ping_parser.extract_pings(
        str(raw_path), channel_filter=data.channel_id
    )
    complex_matrix, _ = ping_parser.get_ping_matrix(pings, data.channel_id)

    datasets.append({
        'name': raw_path.stem,
        'data': data,
        'complex': complex_matrix,
    })
    print(f'  {data.sv_db.shape[1]} pings, {data.sv_db.shape[0]} samples, '
          f'{data.range_array[-1]:.0f}m max range')

print(f'\nLoaded {len(datasets)} echograms.')

## Run all detection methods

For each echogram we run:
1. **Existing ensemble** (simple + edge + gradient with post-processing)
2. **TVG per-ping** — TVG-corrected argmax with leading-edge back-step
3. **DP smoothed** — TVG per-ping + DP temporal smoothing
4. **DP + phase** — DP smoothing with phase coherence bonus
5. **DP + phase + refinement** — active contour polish on top

In [ ]:
results = []

for ds in datasets:
    data = ds['data']
    power_db = data.sv_db
    range_arr = data.range_array
    heave = data.heave if data.heave is not None else np.zeros(power_db.shape[1])

    print(f'Processing {ds["name"]}...')

    # 1. Existing ensemble
    ensemble, outliers, details = detect_bottom_ensemble_pipeline(
        power_db, range_arr,
        heave=heave,
        pulse_duration=data.pulse_duration,
        frequency_hz=data.frequency_hz,
    )

    # 2. TVG per-ping (raw, noisy)
    tvg_raw = detect_bottom_tvg(power_db, range_arr)

    # 3. DP smoothed
    dp_smooth = detect_bottom_dp(power_db, range_arr, smoothness=1.0)

    # 4. DP + phase
    phase_conf = compute_phase_variance(ds['complex'])
    dp_phase = detect_bottom_dp(
        power_db, range_arr, smoothness=1.0,
        phase_confidence=phase_conf, phase_weight=3.0,
    )

    # 5. DP + phase + snake refinement
    dp_refined = refine_bottom_snake(
        power_db, range_arr, dp_phase,
        search_radius_m=3.0, iterations=3,
    )

    results.append({
        'name': ds['name'],
        'power_db': power_db,
        'range_arr': range_arr,
        'phase_conf': phase_conf,
        'ensemble': ensemble,
        'tvg_raw': tvg_raw,
        'dp_smooth': dp_smooth,
        'dp_phase': dp_phase,
        'dp_refined': dp_refined,
    })

print('Done.')

## Visualisation: all methods overlaid on each echogram

In [ ]:
methods = [
    ('ensemble',    'Existing ensemble', 'white',   1.8, (4, 2)),
    ('tvg_raw',     'TVG per-ping',      'yellow',  0.5, ()),
    ('dp_smooth',   'DP smoothed',       'cyan',    1.2, ()),
    ('dp_phase',    'DP + phase',        'lime',    1.2, ()),
    ('dp_refined',  'DP + phase + ref.', 'red',     1.8, ()),
]

for r in results:
    fig, ax = plt.subplots(figsize=(16, 6))
    n_pings = r['power_db'].shape[1]
    ping_idx = np.arange(n_pings)

    ax.imshow(
        r['power_db'], aspect='auto', origin='upper',
        cmap='viridis', vmin=-80, vmax=-20,
        extent=[0, n_pings, r['range_arr'][-1], r['range_arr'][0]],
    )

    for key, label, color, lw, dashes in methods:
        trace = r[key]
        ax.plot(ping_idx, trace, color=color, linewidth=lw,
                alpha=0.85, label=label, dashes=dashes)

    ax.set_xlabel('Ping number')
    ax.set_ylabel('Range (m)')
    ax.set_title(r['name'])
    ax.legend(loc='upper right', fontsize=8, framealpha=0.8)
    plt.tight_layout()
    plt.show()

## Zoomed view: detail comparison

Zoom into a 200-ping window near the center of each echogram to see fine-scale differences between methods.

In [ ]:
ZOOM_WIDTH = 200

for r in results:
    n_pings = r['power_db'].shape[1]
    center = n_pings // 2
    lo = max(0, center - ZOOM_WIDTH // 2)
    hi = min(n_pings, lo + ZOOM_WIDTH)

    # Zoom range from the DP trace (always has values, no NaN)
    dp_slice = r['dp_smooth'][lo:hi]
    depth_center = np.median(dp_slice)
    depth_lo = depth_center - 15
    depth_hi = depth_center + 15

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.imshow(
        r['power_db'], aspect='auto', origin='upper',
        cmap='viridis', vmin=-80, vmax=-20,
        extent=[0, n_pings, r['range_arr'][-1], r['range_arr'][0]],
    )

    for key, label, color, lw, dashes in methods:
        trace = r[key]
        ax.plot(np.arange(n_pings), trace, color=color, linewidth=lw,
                alpha=0.9, label=label, dashes=dashes)

    ax.set_xlim(lo, hi)
    ax.set_ylim(depth_hi, depth_lo)
    ax.set_xlabel('Ping number')
    ax.set_ylabel('Range (m)')
    ax.set_title(f'{r["name"]} — zoomed')
    ax.legend(loc='upper right', fontsize=8, framealpha=0.8)
    plt.tight_layout()
    plt.show()

## Phase coherence maps

Side-by-side: echogram vs phase confidence. The bottom should show up as a bright horizontal band in the phase map.

In [ ]:
for r in results:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    n_pings = r['power_db'].shape[1]
    extent = [0, n_pings, r['range_arr'][-1], r['range_arr'][0]]

    axes[0].imshow(r['power_db'], aspect='auto', origin='upper',
                   cmap='viridis', vmin=-80, vmax=-20, extent=extent)
    axes[0].set_title(f'{r["name"]} — echogram')
    axes[0].set_ylabel('Range (m)')

    im = axes[1].imshow(r['phase_conf'], aspect='auto', origin='upper',
                        cmap='hot', vmin=0, vmax=1, extent=extent)
    axes[1].set_title('Phase coherence (1=bottom, 0=noise)')
    plt.colorbar(im, ax=axes[1], shrink=0.8)

    for ax in axes:
        ax.set_xlabel('Ping number')

    plt.tight_layout()
    plt.show()

## Smoothness and coverage comparison

In [ ]:
print(f'{"File":<30s} {"Method":<22s} {"Coverage %":>10s} {"Smoothness":>12s} {"Mean (m)":>10s}')
print('-' * 90)

for r in results:
    for key, label, *_ in methods:
        trace = r[key]
        valid = ~np.isnan(trace)
        coverage = 100.0 * valid.mean()

        if valid.sum() > 1:
            diffs = np.diff(trace[valid])
            smoothness = np.std(diffs)
            mean_depth = np.mean(trace[valid])
        else:
            smoothness = float('nan')
            mean_depth = float('nan')

        name = r['name'][:28]
        print(f'{name:<30s} {label:<22s} {coverage:>9.1f}% {smoothness:>11.4f}m {mean_depth:>9.1f}m')
    print()

## DP smoothness parameter sensitivity

How does the DP smoothness parameter affect the result? Compare several values on the first echogram.

In [ ]:
r0 = results[0]
smoothness_vals = [0.2, 0.5, 1.0, 2.0, 5.0]
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(smoothness_vals)))

fig, ax = plt.subplots(figsize=(16, 6))
n_pings = r0['power_db'].shape[1]
ax.imshow(
    r0['power_db'], aspect='auto', origin='upper',
    cmap='viridis', vmin=-80, vmax=-20,
    extent=[0, n_pings, r0['range_arr'][-1], r0['range_arr'][0]],
)

# Existing ensemble for reference
ax.plot(np.arange(n_pings), r0['ensemble'], color='white', linewidth=1.5,
        alpha=0.7, label='Ensemble', dashes=(4, 2))

for s, c in zip(smoothness_vals, colors):
    trace = detect_bottom_dp(
        r0['power_db'], r0['range_arr'], smoothness=s,
    )
    ax.plot(np.arange(n_pings), trace, color=c, linewidth=1.2,
            alpha=0.9, label=f'DP s={s}')

ax.set_xlabel('Ping number')
ax.set_ylabel('Range (m)')
ax.set_title(f'{r0["name"]} — DP smoothness sweep')
ax.legend(loc='upper right', fontsize=8, framealpha=0.8)
plt.tight_layout()
plt.show()

## Back-step sensitivity

The back-step parameter controls how far back from the echo peak the detector walks to find the leading edge. Larger values produce shallower detections.

In [ ]:
backstep_vals = [3.0, 6.0, 10.0, 15.0, 20.0]
colors = plt.cm.cool(np.linspace(0.1, 0.9, len(backstep_vals)))

fig, ax = plt.subplots(figsize=(16, 6))
n_pings = r0['power_db'].shape[1]
ax.imshow(
    r0['power_db'], aspect='auto', origin='upper',
    cmap='viridis', vmin=-80, vmax=-20,
    extent=[0, n_pings, r0['range_arr'][-1], r0['range_arr'][0]],
)

ax.plot(np.arange(n_pings), r0['ensemble'], color='white', linewidth=1.5,
        alpha=0.7, label='Ensemble', dashes=(4, 2))

for bs, c in zip(backstep_vals, colors):
    trace = detect_bottom_dp(
        r0['power_db'], r0['range_arr'],
        smoothness=1.0, backstep_db=bs,
    )
    ax.plot(np.arange(n_pings), trace, color=c, linewidth=1.2,
            alpha=0.9, label=f'backstep={bs}dB')

ax.set_xlabel('Ping number')
ax.set_ylabel('Range (m)')
ax.set_title(f'{r0["name"]} — back-step sensitivity')
ax.legend(loc='upper right', fontsize=8, framealpha=0.8)
plt.tight_layout()
plt.show()